# 01 – Exploratory Data Analysis

Analyse the synthetic credit card transaction dataset before model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.join('..', 'data'))
from generate_synthetic_data import generate_synthetic_data

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset

In [ ]:
CSV_PATH = '../data/creditcard.csv'

if not os.path.exists(CSV_PATH):
    print('Generating synthetic dataset ...')
    df = generate_synthetic_data(n_samples=100_000, fraud_ratio=0.02)
    df.to_csv(CSV_PATH, index=False)
    print(f'Saved to {CSV_PATH}')
else:
    df = pd.read_csv(CSV_PATH)

print(f'Shape: {df.shape}')
df.head()

## 2. Basic Inspection

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print('Missing values:')
print(df.isnull().sum().sum())

## 3. Class Distribution

In [ ]:
counts = df['Class'].value_counts()
labels = ['Normal (0)', 'Fraud (1)']
sizes  = [counts[0], counts[1]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, sizes, color=['steelblue', 'tomato'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(sizes):
    axes[0].text(i, v + 100, f'{v:,}', ha='center')

axes[1].pie(sizes, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.show()

print(f'Normal: {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')
print(f'Fraud:  {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')

## 4. Amount Distribution by Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color, label in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Fraud')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=label)

axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')
axes[0].legend()

df.boxplot(column='Amount', by='Class', ax=axes[1],
           notch=True, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Amount Box-plot by Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Amount')

plt.suptitle('')
plt.tight_layout()
plt.show()

print(df.groupby('Class')['Amount'].describe())

## 5. Time-Based Pattern Analysis

In [ ]:
df['hour'] = (df['Time'] / 3600).astype(int) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color, label in [(0, 'steelblue', 'Normal'), (1, 'tomato', 'Fraud')]:
    subset = df[df['Class'] == cls]
    hour_counts = subset['hour'].value_counts().sort_index()
    axes[0].plot(hour_counts.index, hour_counts.values,
                 marker='o', color=color, label=label)

axes[0].set_title('Transactions per Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Count')
axes[0].legend()

df['Time'].hist(bins=48, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Transaction Time Distribution')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Correlation Matrix (V1–V10, Amount)

In [ ]:
cols_to_plot = [f'V{i}' for i in range(1, 11)] + ['Amount', 'Class']
corr = df[cols_to_plot].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix (V1–V10, Amount, Class)')
plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print('=== Summary Statistics by Class ===')
print(df.groupby('Class')[['Amount', 'Time']].describe().T)